# Load libraries

In [ ]:
import os
import gc
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

import joblib

from scipy import signal
from scipy.signal import argrelmax
from scipy.special import expit
from sklearn.metrics import mean_squared_error

# Custom functions

In [ ]:
def sorted_directory_listing(directory):
    items = os.listdir(directory)
    sorted_items = sorted(items)
    return sorted_items

In [ ]:
def lpf(wave, fs=12*60*24, fe=60, n=3):
    nyq = fs / 2.0
    b, a = signal.butter(1, fe/nyq, btype='low')
    for i in range(0, n):
        wave = signal.filtfilt(b, a, wave)
    return wave

# Detect sleep events from predictions

In [ ]:

CHUNK_SIZE = 34560
min_sleep_duration = 30 * 12
activity_tolerance = 30 * 12
order = 6 * 60 * 12

def detect_sleep_events_for_series(predictions, seriesLst, series_lengths):
    all_events = pd.DataFrame(columns=["row_id", "series_id", "step", "event", "score"])
    start_idx = 0
    
    for i, series in tqdm(enumerate(seriesLst), total=len(seriesLst)):
        
        series_length = series_lengths[series.split(".pkl")[0]]
        
        num_full_batches = series_length // CHUNK_SIZE
        remainder = series_length % CHUNK_SIZE
        
        # Extract and concatenate predictions for this series
        series_predictions = []
        for j in range(num_full_batches):
            series_predictions.append(predictions[start_idx + j])
        if remainder > 0:
            series_predictions.append(predictions[start_idx + num_full_batches][:remainder])
        
        series_predictions = np.concatenate(series_predictions)
        start_idx += num_full_batches + (1 if remainder > 0 else 0)
        
        # Apply sigmoid activation
        series_predictions = expit(series_predictions)
        
        # Calculate RMSE before and after applying low-pass filter
        before_RMSE = np.sqrt(mean_squared_error(series_predictions, np.zeros_like(series_predictions)))
        series_predictions = lpf(series_predictions[:, 0])
        after_RMSE = np.sqrt(mean_squared_error(series_predictions, np.zeros_like(series_predictions)))
        
        # Adjust predictions based on decay ratio
        decay_ratio = before_RMSE / after_RMSE
        series_predictions *= decay_ratio
        
        # Find local maxima for onset and wakeup
        onset_indices = argrelmax(series_predictions, order=order)[0]
        wakeup_indices = argrelmax(-series_predictions, order=order)[0]
        
        final_onsets = []
        final_wakeups = []
        
        for onset in onset_indices:
            valid_wakeups = wakeup_indices[wakeup_indices > onset]
            if valid_wakeups.size == 0:
                continue
            
            wakeup = valid_wakeups[0]
            if (wakeup - onset) >= min_sleep_duration:
                # Combine adjacent sleep periods
                while len(final_wakeups) > 0 and (onset - final_wakeups[-1]) <= activity_tolerance:
                    onset = final_onsets.pop()
                    final_wakeups.pop()
                
                final_onsets.append(onset)
                final_wakeups.append(wakeup)
                wakeup_indices = wakeup_indices[wakeup_indices > wakeup]
        
        series_events = []
        for onset, wakeup in zip(final_onsets, final_wakeups):
            
            # Onset
            series_events.append({"row_id": len(series_events), 
                                  "series_id": series, 
                                  "step": onset, 
                                  "event": "onset", 
                                  "score": series_predictions[onset]})
            
            # Wakeup
            series_events.append({"row_id": len(series_events), 
                                  "series_id": series, 
                                  "step": wakeup, 
                                  "event": "wakeup", 
                                  "score": series_predictions[wakeup]})
        
        all_events = pd.concat([all_events, pd.DataFrame(series_events)], ignore_index=True)
    
    return all_events



In [ ]:
predictions = np.load("/kaggle/input/fork-of-zzzs-lstm-training-tf/val_predictions_fold_1.npy")
print(f"Val preds: {predictions.shape}")

seriesLst = sorted_directory_listing("/kaggle/input/zzzs-numpyfolds-fold-1/fold1/")
print(f"seriesLst: {len(seriesLst)}")

series_lengths = joblib.load("/kaggle/input/zzzs-valserieslens/val_serieslens_fold1.pkl")

submission = detect_sleep_events_for_series(predictions, seriesLst, series_lengths)
print(submission)
